# Notebook 03 — What a proton track looks like

This notebook draws **one proton's track, collision by collision**, at two
energies: a slow proton (15 keV/u) and a fast one (500 keV/u). It is the
most direct picture of what "event-by-event" transport means: every dot
below is one simulated collision.

**What you will see**

- The proton's path, coloured by its **charge state**: blue while it is a
  proton (H⁺, `H1+`), orange while it is a neutral hydrogen atom (H⁰,
  `H0`), after it has captured an electron.
- A marker at every collision, coloured by **process**: excitation,
  ionization, capture, loss or elastic scattering.
- A black dot wherever a **δ-electron** (an electron knocked out by an
  ionization) deposited its energy.

**Before you start.** Read notebook 01 first, at least up to §3; it
explains the cross sections and the transport loop used here. This
notebook runs in about 10 seconds. Its two figures are also saved in
`notebooks/output_03/`.

**Two things to keep in mind when reading the plots**

1. **Electrons are drawn as points, not tracks.** In this code a
   δ-electron deposits all its energy where it is created; it is not
   transported. So electrons appear as dots sitting on the proton's path.
   In reality they would fly off sideways, some for tens of nm.
2. **The plots are 2D projections with very different axis scales.** Depth
   $z$ (along the beam) is horizontal; one transverse coordinate $x$ is
   vertical. The vertical axis is stretched enormously relative to the
   horizontal one, so the tracks look wiggly. In true proportions they are
   almost perfectly straight lines.

In [ ]:
# Google Colab only: download the repository and install the package.
# On your own computer (after ./setup.sh) this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/ion-track-mc"):
        subprocess.run(["git", "clone", "-q", "https://github.com/juanmontiz/ion-track-mc",
                        "/content/ion-track-mc"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/ion-track-mc"], check=True)
    os.chdir("/content/ion-track-mc/notebooks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from ion_track_mc import constants
from ion_track_mc.physics.channels import Channel
from ion_track_mc.physics.cross_section import (
    TabulatedCrossSectionSet,
    load_cross_section_table,
    load_mean_energy_table,
)
from ion_track_mc.physics.medium import WATER
from ion_track_mc.transport.engine import TrackEngine
from ion_track_mc.transport.particle import Particle
from ion_track_mc.analysis.visualization import plot_track, save_figure, use_paper_style

use_paper_style()

OUT_DIR = Path("output_03")  # every figure below is also saved here as PNG + PDF

## 1. Load the cross sections and the energy-loss tables

Two kinds of tabulated data define the physics (see
`ion_track_mc/data/README_data_format.md`):

- **Cross sections** $\sigma_c(E)$ (`TCS` files), one per process $c$ and
  charge state. Together they decide *how far* the projectile flies between
  collisions (mean free path $1/N\sigma_{\text{tot}}$) and *which* process
  happens (probability $\sigma_c/\sigma_{\text{tot}}$).
- **Mean energies**, which decide *how much* energy each collision costs:
  `Edepmeantot` is the energy left at the collision site, and `Eemeantot`
  is the kinetic energy given to the ejected δ-electron in an ionization.

H⁺ (`q = 1`) can undergo ionization, excitation, capture and elastic
scattering; H⁰ (`q = 0`) ionization, excitation, loss and elastic
scattering. The dictionaries below are keyed by `(q, channel)` for that
reason. This cell is the same in all three notebooks.

In [ ]:
DATA_DIR = Path("..") / "ion_track_mc" / "data" / "cross_sections"
FILES = {
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_TCS_CDW-EIS.dat",
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_TCS_Miller-Green.dat",
    (1, Channel.CAPTURE): "H1+_+_H2O_capture_TCS_CDW-EIS.dat",
    (1, Channel.ELASTIC): "H1+_+_H2O_elastic_TCS_Classical.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_TCS_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_TCS_Miller-Green.dat",
    (0, Channel.LOSS): "H0_+_H2O_loss_TCS_Miller-Green.dat",
    (0, Channel.ELASTIC): "H0_+_H2O_elastic_TCS_Classical.dat",
}
tables = {k: load_cross_section_table(DATA_DIR / f) for k, f in FILES.items()}
xs = TabulatedCrossSectionSet(tables)

EDEP_FILES = {
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (1, Channel.CAPTURE):    "H1+_+_H2O_capture_Edepmeantot_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (0, Channel.LOSS):       "H0_+_H2O_loss_Edepmeantot_constant.dat",
}
EEMEAN_FILES = {
    1: "H1+_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
    0: "H0_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
}
edep_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EDEP_FILES.items()}
eemean_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EEMEAN_FILES.items()}

## 2. A slow proton: 15 keV/u

Below we create a `TrackEngine` (the collision loop), start a proton
(`q = 1`) at the origin moving along $+z$, and call `transport()`. That one
call runs the whole track: thousands of collisions, until the energy drops
below the lowest tabulated energy (8 keV).

`entry_low_copy` keeps the starting state, because `transport()` updates
the particle in place and the plot needs to know where the track began.
`seed=1` fixes the random numbers, so you get exactly the same track every
time you run the cell; change the seed to see a different track.

**What to expect.** At 15 keV/u the proton is *below* the stopping-power
maximum: in this model the stopping power (energy lost per nm) peaks at
about 70 keV/u, and at 15 keV/u it is about 60 eV/nm. Only about 7 keV
can be lost before the proton reaches the 8 keV cut-off where tracking
stops, so the track is short, about 130 nm.

The most important feature at this energy is **charge exchange**. Electron
capture is very likely for slow protons. In charge-state equilibrium the
projectile spends about **82% of its path as neutral H⁰** at 15 keV/u,
against 15% at 100 keV/u and 0.2% at 500 keV/u. Expect the track to be
full of capture and loss events.

In [ ]:
E_LOW_EV = 15.0 * 1.0e3 * constants.PROTON_NUCLEON_COUNT
engine_low = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=1)
entry_low = Particle(kind="proton", position=np.zeros(3), direction=np.array([0.0, 0.0, 1.0]),
                      energy=E_LOW_EV, q=1)
entry_low_copy = Particle(kind=entry_low.kind, position=entry_low.position.copy(),
                           direction=entry_low.direction.copy(), energy=entry_low.energy, q=entry_low.q)
engine_low.transport(entry_low)
fig, ax = plt.subplots(figsize=(7, 2))
plot_track(entry_low_copy, engine_low, ax=ax, marker_size=1, electron_marker_size=1,
           line_width=1.2, aspect="auto",
           channel_marker_sizes={Channel.IONIZATION: 3, Channel.CAPTURE: 5},
           legend_fontsize=6)
save_figure(fig, OUT_DIR, "01_track_15keV_u")
plt.show()

**How to read this figure.** The track is about 130 nm long and drifts
about 3 nm sideways. Remember that the vertical axis is stretched: the
path is essentially straight.

- **Capture (purple) and loss (brown) markers are everywhere.** The
  projectile keeps switching between H⁺ and H⁰, many times over a few nm.
  This is why the code treats the charge state $q$ as part of the
  particle's state and changes it at every capture or loss.
- **Black dots (δ-electron deposits) are densely packed.** Many collisions
  are ionizations, and every ionization leaves an electron's energy right
  on the track.
- **Elastic events (cyan)** slightly change the direction. They are what
  makes the track bend at all.

Note that this track is *not* the densest possible: its average energy
loss, about 7 keV over 130 nm ≈ 54 eV/nm, is *below* the stopping-power
maximum. What makes it special is the constant charge exchange, not the
density of energy deposition.

## 3. A fast proton: 500 keV/u

Same procedure, now at 500 keV/u, well above the stopping-power maximum.
Here the stopping power is lower, about 43 eV/nm at the start. There is
also much more energy to lose, so the track is much longer, several µm.
Capture is now rare: the projectile is a bare proton almost all the time.

The marker sizes are reduced in this cell; with tens of thousands of
collisions, full-size markers would cover the line completely.

In [ ]:
E_HIGH_EV = 500.0 * 1.0e3 * constants.PROTON_NUCLEON_COUNT
engine_high = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=1)
entry_high = Particle(kind="proton", position=np.zeros(3), direction=np.array([0.0, 0.0, 1.0]),
                       energy=E_HIGH_EV, q=1)
entry_high_copy = Particle(kind=entry_high.kind, position=entry_high.position.copy(),
                            direction=entry_high.direction.copy(), energy=entry_high.energy, q=entry_high.q)
engine_high.transport(entry_high)
fig, ax = plt.subplots(figsize=(7, 2))
plot_track(entry_high_copy, engine_high, ax=ax, marker_size=2, electron_marker_size=0,
           line_width=1.2, aspect="auto",
           channel_marker_sizes={Channel.IONIZATION: 0.1, Channel.CAPTURE: 1.7, Channel.EXCITATION: 0.1, Channel.LOSS: 0.1, Channel.ELASTIC: 0},
           legend_fontsize=5)
save_figure(fig, OUT_DIR, "02_track_500keV_u")
plt.show()

**How to read this figure.** The track is about 7.6 µm long. The
sideways excursions, between −8 and +18 nm, are tiny compared with that:
the vertical axis is stretched by a factor of about 80 (about 12 in the
15 keV/u figure). Note three things:

- **The proton loses energy gradually and keeps going straight.** Each
  collision costs only tens of eV out of 500 keV, and a fast, heavy
  projectile is deflected by only a tiny angle in each elastic collision.
- **Capture markers (purple) are rare at the start and become frequent
  near the end.** As the proton slows down it enters the energy range
  where capture is likely, and the last part of the track looks like the
  15 keV/u track above.
- **The direction changes most in the last few hundred nm.** Slow protons
  are deflected more easily, so the end of the track bends.

**Comparing the two tracks.** The fast proton loses energy slowly at
first (about 43 eV/nm). As it slows down, its stopping power rises,
passes through its maximum near 70 keV/u, and only falls again in the last
stretch, below that energy, which is the regime of the 15 keV/u track.
The **Bragg peak** therefore sits where the stopping power is largest, near
(but not at) the very end of the range. Notebook 01 §8 and notebook 02
build that curve from many tracks.